# Exploring the bundled material library

Peritheos constructs its executable catalog directly from the versioned `.eosmat`
library. This notebook discovers the full library, searches executable objects,
summarizes model coverage, and inspects the underlying raw document.

All operations are offline: the material documents, schema, and audit metadata
are installed with the package.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from peritheos import (
    get_material,
    get_material_document,
    list_eos_records,
    list_material_documents,
    list_materials,
    search_materials,
)

plt.style.use("seaborn-v0_8-whitegrid")

material_ids = list_material_documents()
materials = list_materials()
executable_records = list_eos_records()
documents = [get_material_document(identifier) for identifier in material_ids]
records = [record for document in documents for record in document["eos_records"]]

print(f"Executable materials: {len(materials)}")
print(f"Executable EOS records: {len(executable_records)}")
print(f"Thermal records: {sum(record.is_thermal for record in executable_records)}")
print("First ten identifiers:", ", ".join(material_ids[:10]))

## Summarize equation coverage

The equation discriminator lives inside each record's `eos` component. Thermal
components are counted separately because they compose with a reference
isotherm rather than replacing it.

In [ ]:
isothermal_counts = Counter(record["eos"]["model"] for record in records)
thermal_counts = Counter(
    record["thermal"]["model"]
    for record in records
    if record.get("thermal") is not None
)

print("Isothermal models:")
for model, count in isothermal_counts.most_common():
    print(f"  {model:42s} {count:3d}")

print("\nThermal models:")
for model, count in thermal_counts.most_common():
    print(f"  {model:42s} {count:3d}")

In [ ]:
model_labels = []
model_values = []
for model, count in (isothermal_counts + thermal_counts).most_common(10):
    model_labels.append(model)
    model_values.append(count)

fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.barh(model_labels[::-1], model_values[::-1])
ax.set(xlabel="Number of records", title="Ten most common EOS components")
plt.show()

## Search by scientific metadata

The typed search API filters identity, equation, reference, capability,
uncertainty, validation, and calibration metadata while returning executable
objects. This query finds every material containing gold.

In [ ]:
matches = [material.to_eosmat() for material in search_materials(formula="Au")]
for document in matches:
    print(
        f"{document['identifier']}: {document['name']} "
        f"({len(document['eos_records'])} records)"
    )
    for record in document["eos_records"]:
        thermal = record.get("thermal")
        thermal_name = thermal["model"] if thermal is not None else "none"
        validation = record["scientific_validation"]["status"]
        print(
            f"  {record['identifier']}\n"
            f"    cold={record['eos']['model']}, thermal={thermal_name}\n"
            f"    validation={validation}, DOI={record['reference'].get('doi')}"
        )

## Inspect the shared structural and EOS document

The crystallographic information and EOS records belong to one flat material
document. Peritheos preserves structure fields for interoperability even
though pressure calculations use only the material's cell definition and EOS.

In [ ]:
gold_document = get_material_document("gold")
print("Document format:", gold_document["format"], gold_document["format_version"])
print("Formula:", gold_document["formula"])
print("Symmetry:", gold_document.get("symmetry"))
print("Space group:", gold_document.get("space_group"))
print("Formula units per cell:", gold_document.get("formula_units_per_cell"))
print("Lattice:", gold_document.get("lattice"))
print("Atom sites:", gold_document.get("atom_sites"))

## Construct and use an executable record

The normal lookup already returns objects built from validated `.eosmat` model
discriminators. Selecting by stable record identifier keeps the exact source and
parameterization attached to the calculation.

In [ ]:
gold = get_material("gold")
print("Executable records:")
for record in gold.eos_records:
    print(f"  {record.identifier}: {record.name}")

fei = gold.get_eos_record("gold_fei_2007_vinet_2")
pressure = fei.pressure(volume=55.0, temperature=1800.0)
print(f"\nFei Au at V=55.0 A^3 and T=1800 K: {pressure:.3f} GPa")
print("Within stored calibration range:", fei.within_calibration_range(55.0, 1800.0))
print("Source DOI:", fei.reference.doi)

## Takeaways

1. Use `list_materials`, `list_eos_records`, and typed search for calculations.
2. Use `get_material_document` only when raw exchange data is needed.
3. Stable material and record identifiers make calculations reproducible.
4. Provenance, validation, units, structure, and numerical parameters travel
   together in the `.eosmat` document.